<a href="https://colab.research.google.com/github/Ryan56-56/onix/blob/main/onix(4).ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [50]:
import os
os.environ["ONNX_NO_EXTERNAL_DATA"] = "1"


In [65]:
!pip install onnx onnxruntime scikit-learn joblib onnxscript skl2onnx

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 317.2/317.2 kB 7.2 MB/s eta 0:00:00


In [41]:
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from torch.utils.data import TensorDataset, DataLoader
import joblib

In [42]:
df = pd.read_csv("/content/drive/MyDrive/data.csv", sep=';')
df.head()

,Marital status,Application mode,Application order,Course,Daytime/evening attendance\t,Previous qualification,Previous qualification (grade),Nacionality,Mother's qualification,Father's qualification,...,Curricular units 2nd sem (credited),Curricular units 2nd sem (enrolled),Curricular units 2nd sem (evaluations),Curricular units 2nd sem (approved),Curricular units 2nd sem (grade),Curricular units 2nd sem (without evaluations),Unemployment rate,Inflation rate,GDP,Target
0,1,17,5,171,1,1,122.0,1,19,12,...,0,0,0,0,0.000000,0,10.8,1.4,1.74,Dropout
1,1,15,1,9254,1,1,160.0,1,1,3,...,0,6,6,6,13.666667,0,13.9,-0.3,0.79,Graduate
2,1,1,5,9070,1,1,122.0,1,37,37,...,0,6,0,0,0.000000,0,10.8,1.4,1.74,Dropout
3,1,17,2,9773,1,1,122.0,1,38,37,...,0,6,10,5,12.400000,0,9.4,-0.8,-3.12,Graduate
4,2,39,1,8014,0,1,100.0,1,37,38,...,0,6,6,6,13.000000,0,13.9,-0.3,0.79,Graduate


In [43]:
FEATURES = [
    'Marital status','Application mode','Application order','Course',
    'Daytime/evening attendance\t','Previous qualification',
    'Previous qualification (grade)','Nacionality',"Mother's qualification",
    "Father's qualification","Mother's occupation","Father's occupation",
    'Admission grade','Displaced','Educational special needs','Debtor',
    'Tuition fees up to date','Gender','Scholarship holder',
    'Age at enrollment','International',
    'Curricular units 1st sem (credited)','Curricular units 1st sem (enrolled)',
    'Curricular units 1st sem (evaluations)','Curricular units 1st sem (approved)',
    'Curricular units 1st sem (grade)','Curricular units 1st sem (without evaluations)',
    'Curricular units 2nd sem (credited)','Curricular units 2nd sem (enrolled)',
    'Curricular units 2nd sem (evaluations)','Curricular units 2nd sem (approved)',
    'Curricular units 2nd sem (grade)','Curricular units 2nd sem (without evaluations)',
    'Unemployment rate','Inflation rate','GDP'
]

TARGET = "Target"

In [31]:
X = df[FEATURES].values.astype(np.float32)
y = pd.factorize(df[TARGET])[0].astype(np.float32).reshape(-1, 1)

In [32]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

In [33]:
scaler_x = StandardScaler()
scaler_y = StandardScaler()

X_train_scaled = scaler_x.fit_transform(X_train)
X_test_scaled  = scaler_x.transform(X_test)

y_train_scaled = scaler_y.fit_transform(y_train)
y_test_scaled  = scaler_y.transform(y_test)

In [34]:
train_ds = TensorDataset(
    torch.tensor(X_train_scaled, dtype=torch.float32),
    torch.tensor(y_train_scaled, dtype=torch.float32)
)

train_dl = DataLoader(train_ds, batch_size=32, shuffle=True)

In [44]:
class StudentNet(nn.Module):
    def __init__(self):
        super().__init__()
        self.fc1 = nn.Linear(36, 64) # Corrected input dimension to 36
        self.fc2 = nn.Linear(64, 64)
        self.fc3 = nn.Linear(64, 1)
        self.act = nn.ReLU()

    def forward(self, x):
        x = self.act(self.fc1(x))
        x = self.act(self.fc2(x))
        return self.fc3(x)

model = StudentNet()
optimizer = optim.Adam(model.parameters(), lr=0.001)
loss_fn = nn.MSELoss()

In [45]:
for epoch in range(200):
    for xb, yb in train_dl:
        optimizer.zero_grad()
        pred = model(xb)
        loss = loss_fn(pred, yb)
        loss.backward()
        optimizer.step()

    if epoch % 50 == 0:
        print(f"Epoch {epoch} Loss {loss.item():.4f}")

Epoch 0 Loss 0.8063
Epoch 50 Loss 0.2799
Epoch 100 Loss 0.0814
Epoch 150 Loss 0.0361


In [51]:
torch.onnx.export(
    model,
    dummy,
    "model.onnx",
    input_names=["input"],
    output_names=["output"],
    opset_version=17,
    export_params=True,
    do_constant_folding=True,
    keep_initializers_as_inputs=False
)


/tmp/ipykernel_8855/233600418.py:1: UserWarning: Exporting a model while it is in training mode. Please ensure that this is intended, as it may lead to different behavior during inference. Calling model.eval() before export is recommended.
  torch.onnx.export(
W0418 03:13:03.126000 8855 torch/onnx/_internal/exporter/_compat.py:125] Setting ONNX exporter to use operator set version 18 because the requested opset_version 17 is a lower version than we have implementations for. Automatic version conversion will be performed, which may not be successful at converting to the requested version. If version conversion is unsuccessful, the opset version of the exported model will be kept at 18. Please consider setting opset_version >=18 to leverage latest ONNX features


[torch.onnx] Obtain model graph for `StudentNet([...]` with `torch.export.export(..., strict=False)`...
[torch.onnx] Obtain model graph for `StudentNet([...]` with `torch.export.export(..., strict=False)`... ✅
[torch.onnx] Run decomposition...


/usr/lib/python3.12/copyreg.py:99: FutureWarning: `isinstance(treespec, LeafSpec)` is deprecated, use `isinstance(treespec, TreeSpec) and treespec.is_leaf()` instead.
  return cls.__new__(cls, *args)


[torch.onnx] Run decomposition... ✅
[torch.onnx] Translate the graph into ONNX...
[torch.onnx] Translate the graph into ONNX... ✅


ONNXProgram(
    model=
        <
            ir_version=10,
            opset_imports={'': 17},
            producer_name='pytorch',
            producer_version='2.10.0+cpu',
            domain=None,
            model_version=None,
        >
        graph(
            name=main_graph,
            inputs=(
                %"input"<FLOAT,[1,36]>
            ),
            outputs=(
                %"output"<FLOAT,[1,1]>
            ),
            initializers=(
                %"fc1.bias"<FLOAT,[64]>{TorchTensor(...)},
                %"fc2.bias"<FLOAT,[64]>{TorchTensor(...)},
                %"fc3.weight"<FLOAT,[1,64]>{TorchTensor(...)},
                %"fc3.bias"<FLOAT,[1]>{TorchTensor<FLOAT,[1]>(Parameter containing: tensor([0.0774], requires_grad=True), name='fc3.bias')},
                %"fc1.weight"<FLOAT,[64,36]>{TorchTensor(...)},
                %"fc2.weight"<FLOAT,[64,64]>{TorchTensor(...)}
            ),
        ) {
            0 |  # node_linear
                 %"linear"<

In [52]:
import onnx
onnx.save_model = lambda model, f, **kwargs: onnx._serialize(model, f, save_as_external_data=False)


In [54]:
import onnx
from onnx import numpy_helper

# Export normally first
torch.onnx.export(
    model,
    dummy,
    "model_temp.onnx",
    input_names=["input"],
    output_names=["output"],
    opset_version=17,
    export_params=True,
    do_constant_folding=True
)

# Load the model
model_onnx = onnx.load("model_temp.onnx")

# Force embed all external data
onnx.save(
    model_onnx,
    "model.onnx",
    save_as_external_data=False
)

print("Saved single-file model.onnx")


/tmp/ipykernel_8855/2176636775.py:5: UserWarning: Exporting a model while it is in training mode. Please ensure that this is intended, as it may lead to different behavior during inference. Calling model.eval() before export is recommended.
  torch.onnx.export(
W0418 03:14:50.152000 8855 torch/onnx/_internal/exporter/_compat.py:125] Setting ONNX exporter to use operator set version 18 because the requested opset_version 17 is a lower version than we have implementations for. Automatic version conversion will be performed, which may not be successful at converting to the requested version. If version conversion is unsuccessful, the opset version of the exported model will be kept at 18. Please consider setting opset_version >=18 to leverage latest ONNX features


[torch.onnx] Obtain model graph for `StudentNet([...]` with `torch.export.export(..., strict=False)`...
[torch.onnx] Obtain model graph for `StudentNet([...]` with `torch.export.export(..., strict=False)`... ✅
[torch.onnx] Run decomposition...


/usr/lib/python3.12/copyreg.py:99: FutureWarning: `isinstance(treespec, LeafSpec)` is deprecated, use `isinstance(treespec, TreeSpec) and treespec.is_leaf()` instead.
  return cls.__new__(cls, *args)


[torch.onnx] Run decomposition... ✅
[torch.onnx] Translate the graph into ONNX...
[torch.onnx] Translate the graph into ONNX... ✅
Saved single-file model.onnx


In [47]:
joblib.dump(scaler_x, "scaler_x.pkl")
joblib.dump(scaler_y, "scaler_y.pkl")

['scaler_y.pkl']

In [57]:
for i, col in enumerate(df.columns):
    print(i, col)


0 Marital status
1 Application mode
2 Application order
3 Course
4 Daytime/evening attendance	
5 Previous qualification
6 Previous qualification (grade)
7 Nacionality
8 Mother's qualification
9 Father's qualification
10 Mother's occupation
11 Father's occupation
12 Admission grade
13 Displaced
14 Educational special needs
15 Debtor
16 Tuition fees up to date
17 Gender
18 Scholarship holder
19 Age at enrollment
20 International
21 Curricular units 1st sem (credited)
22 Curricular units 1st sem (enrolled)
23 Curricular units 1st sem (evaluations)
24 Curricular units 1st sem (approved)
25 Curricular units 1st sem (grade)
26 Curricular units 1st sem (without evaluations)
27 Curricular units 2nd sem (credited)
28 Curricular units 2nd sem (enrolled)
29 Curricular units 2nd sem (evaluations)
30 Curricular units 2nd sem (approved)
31 Curricular units 2nd sem (grade)
32 Curricular units 2nd sem (without evaluations)
33 Unemployment rate
34 Inflation rate
35 GDP
36 Target


In [61]:
import pandas as pd

df = pd.read_csv('/content/drive/MyDrive/data.csv', sep=';')

In [62]:
X = df.drop(columns=["Target"])
y = df["Target"]


In [63]:
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

model = RandomForestClassifier()
model.fit(X_train, y_train)


RandomForestClassifier()

In [66]:
import skl2onnx
from skl2onnx import convert_sklearn
from skl2onnx.common.data_types import FloatTensorType

initial_type = [('input', FloatTensorType([None, X.shape[1]]))]  # X.shape[1] = 36
onnx_model = convert_sklearn(model, initial_types=initial_type)

with open("model.onnx", "wb") as f:
    f.write(onnx_model.SerializeToString())
